In [1]:
from langchain_experimental.text_splitter import SemanticChunker
from langchain_community.embeddings import HuggingFaceEmbeddings
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from dotenv import load_dotenv
import os

load_dotenv()
MINIML_MODEL_PATH = os.getenv("MINIML_MODEL_PATH")

In [2]:
# Usage
embeddings = HuggingFaceEmbeddings(model_name=MINIML_MODEL_PATH)

class ImprovedSemanticChunker:
    def __init__(self, embeddings_model, similarity_threshold=0.75, min_chunk_size=2):
        self.embeddings = embeddings_model
        self.similarity_threshold = similarity_threshold
        self.min_chunk_size = min_chunk_size
    
    def split_into_sentences(self, text):
        """Split text into sentences"""
        # Handle different sentence endings
        text = text.replace('\n', ' ')
        sentences = []
        for sent in text.split('. '):
            if sent.strip():
                sentences.append(sent.strip() + '.')
        return sentences
    
    def get_embeddings_batch(self, sentences):
        """Get embeddings for multiple sentences"""
        return [self.embeddings.embed_query(sent) for sent in sentences]
    
    def semantic_chunking(self, text):
        """Main chunking logic"""
        sentences = self.split_into_sentences(text)
        
        if len(sentences) <= self.min_chunk_size:
            return [text]
        
        # Get embeddings for all sentences
        embeddings_list = self.get_embeddings_batch(sentences)
        
        chunks = []
        current_chunk = [sentences[0]]
        current_embeddings = [embeddings_list[0]]
        
        for i in range(1, len(sentences)):
            # Calculate similarity with the last sentence in current chunk
            last_emb = current_embeddings[-1]
            current_emb = embeddings_list[i]
            
            similarity = cosine_similarity([last_emb], [current_emb])[0][0]
            
            # Also check similarity with the first sentence of chunk for context
            first_emb = current_embeddings[0]
            similarity_with_first = cosine_similarity([first_emb], [current_emb])[0][0]
            
            # Use average similarity for decision
            avg_similarity = (similarity + similarity_with_first) / 2
            
            if avg_similarity >= self.similarity_threshold:
                # Semantically similar - add to current chunk
                current_chunk.append(sentences[i])
                current_embeddings.append(embeddings_list[i])
            else:
                # Semantic shift - create new chunk if current chunk meets minimum size
                if len(current_chunk) >= self.min_chunk_size:
                    chunks.append(" ".join(current_chunk))
                    current_chunk = [sentences[i]]
                    current_embeddings = [embeddings_list[i]]
                else:
                    # If chunk is too small, force include this sentence
                    current_chunk.append(sentences[i])
                    current_embeddings.append(embeddings_list[i])
        
        # Add the last chunk
        if current_chunk:
            chunks.append(" ".join(current_chunk))
        
        return chunks



C:\Users\scientist\AppData\Local\Temp\ipykernel_1136\2186282115.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name=MINIML_MODEL_PATH)
D:\NewAgentic\RAG_Terms\venv_chunk\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6007.53it/s]


In [6]:
text = """
Python is a programming language. It is widely used in AI.
Machine learning is a subset of AI. AI is artificial intelligence. It focuses on AI models. there is change in stock market, if this happens.
The stock market crashed yesterday. Investors are worried.
"""

# Test different thresholds
for threshold in [0.65, 0.70, 0.75, 0.80]:
    print(f"\n{'='*50}")
    print(f"SIMILARITY THRESHOLD: {threshold}")
    print('='*50)
    
    chunker = ImprovedSemanticChunker(
        embeddings_model=embeddings,
        similarity_threshold=threshold,
        min_chunk_size=2  # Minimum sentences per chunk
    )
    
    chunks = chunker.semantic_chunking(text)
    
    for i, chunk in enumerate(chunks):
        print(f"\nChunk {i+1}:\n{chunk}")
    
    print(f"\nTotal chunks: {len(chunks)}")


SIMILARITY THRESHOLD: 0.65

Chunk 1:
Python is a programming language. It is widely used in AI.

Chunk 2:
Machine learning is a subset of AI. AI is artificial intelligence.

Chunk 3:
It focuses on AI models. there is change in stock market, if this happens.

Chunk 4:
The stock market crashed yesterday. Investors are worried.

Total chunks: 4

SIMILARITY THRESHOLD: 0.7

Chunk 1:
Python is a programming language. It is widely used in AI.

Chunk 2:
Machine learning is a subset of AI. AI is artificial intelligence.

Chunk 3:
It focuses on AI models. there is change in stock market, if this happens.

Chunk 4:
The stock market crashed yesterday. Investors are worried.

Total chunks: 4

SIMILARITY THRESHOLD: 0.75

Chunk 1:
Python is a programming language. It is widely used in AI.

Chunk 2:
Machine learning is a subset of AI. AI is artificial intelligence.

Chunk 3:
It focuses on AI models. there is change in stock market, if this happens.

Chunk 4:
The stock market crashed yesterday. Invest

In [7]:
# Best configuration for your use case
splitter = SemanticChunker(
    embeddings=embeddings,
    breakpoint_threshold_type="interquartile",  # More stable than percentile
    breakpoint_threshold_amount=0.6  # Higher threshold = fewer chunks
)

chunks = splitter.split_text(text)
print("OPTIMAL CONFIGURATION:")
for i, chunk in enumerate(chunks):
    if chunk.strip():
        print(f"\nChunk {i+1}:\n{chunk}")

OPTIMAL CONFIGURATION:

Chunk 1:

Python is a programming language. It is widely used in AI. Machine learning is a subset of AI. AI is artificial intelligence.

Chunk 2:
It focuses on AI models. there is change in stock market, if this happens. The stock market crashed yesterday. Investors are worried. 


NameError: name 'ContextAwareSemanticChunker' is not defined